In [18]:
import argparse
import math
import pickle
import random
from collections import defaultdict
from pathlib import Path
import numpy as np

In [19]:
def read_ratings(path):
    rows = []
    with open(path, "r", encoding="latin-1") as f:
        for line in f:
            user_id, item_id, rating, timestamp = line.strip().split("\t")
            rows.append((int(user_id), int(item_id), float(rating), int(timestamp)))
    return rows


def read_items(path):
    titles = {}
    with open(path, "r", encoding="latin-1") as f:
        for line in f:
            parts = line.rstrip("\n").split("|")
            titles[int(parts[0])] = parts[1]
    return titles

def split_validation_from_base(base_rows, valid_ratio=0.1, seed=42):
    rng = random.Random(seed)
    by_user = defaultdict(list)
    for row in base_rows:
        by_user[row[0]].append(row)
    train_rows, valid_rows = [], []
    for user_rows in by_user.values():
        shuffled = list(user_rows)
        rng.shuffle(shuffled)
        n_valid = max(1, int(len(shuffled) * valid_ratio)) if len(shuffled) >= 5 else 0
        valid_rows.extend(shuffled[:n_valid])
        train_rows.extend(shuffled[n_valid:])
    return train_rows, valid_rows


In [20]:
class ItemKNNRecommender:
    def __init__(self, k=40, shrinkage=25.0, min_candidate_popularity=5):
        self.k = k
        self.shrinkage = shrinkage
        self.min_candidate_popularity = min_candidate_popularity

    def fit(self, rows):
        self.users = sorted({u for u, _, _, _ in rows})
        self.items = sorted({i for _, i, _, _ in rows})
        self.user_to_idx = {u: idx for idx, u in enumerate(self.users)}
        self.item_to_idx = {i: idx for idx, i in enumerate(self.items)}
        n_users, n_items = len(self.users), len(self.items)
        self.ratings = np.full((n_users, n_items), np.nan, dtype=np.float32)
        self.user_rated_items = defaultdict(list)
        self.item_popularity = defaultdict(int)
        for user_id, item_id, rating, _ in rows:
            uidx = self.user_to_idx[user_id]
            iidx = self.item_to_idx[item_id]
            self.ratings[uidx, iidx] = rating
            self.user_rated_items[user_id].append((item_id, rating))
            self.item_popularity[item_id] += 1
        self.global_mean = float(np.nanmean(self.ratings))
        self.user_means = np.nanmean(self.ratings, axis=1)
        self.item_means = np.nanmean(self.ratings, axis=0)
        self.user_means = np.where(np.isnan(self.user_means), self.global_mean, self.user_means)
        self.item_means = np.where(np.isnan(self.item_means), self.global_mean, self.item_means)
        mask = ~np.isnan(self.ratings)
        centered = np.where(mask, self.ratings - self.user_means[:, None], 0.0)
        numerator = centered.T @ centered
        norms = np.sqrt(np.sum(centered * centered, axis=0))
        denominator = norms[:, None] * norms[None, :]
        similarity = np.divide(numerator, denominator,
            out=np.zeros_like(numerator, dtype=np.float32), where=denominator > 0)
        co_counts = mask.astype(np.float32).T @ mask.astype(np.float32)
        significance = co_counts / (co_counts + self.shrinkage)
        self.similarity = similarity * significance
        np.fill_diagonal(self.similarity, 0.0)
        self.neighbors = {}
        for item_idx in range(n_items):
            sims = self.similarity[item_idx]
            order = np.argsort(np.abs(sims))[::-1]
            order = [idx for idx in order if sims[idx] != 0.0][: self.k]
            self.neighbors[item_idx] = order
        return self

    def predict(self, user_id, item_id):
        if user_id not in self.user_to_idx:
            return self._clip(self.item_mean(item_id))
        if item_id not in self.item_to_idx:
            return self._clip(self.user_mean(user_id))
        uidx = self.user_to_idx[user_id]
        target_idx = self.item_to_idx[item_id]
        baseline = self.user_means[uidx]
        numerator = denominator = 0.0
        for neighbor_idx in self.neighbors[target_idx]:
            rating = self.ratings[uidx, neighbor_idx]
            if np.isnan(rating):
                continue
            sim = float(self.similarity[target_idx, neighbor_idx])
            numerator += sim * (float(rating) - baseline)
            denominator += abs(sim)
        if denominator == 0.0:
            return self._clip(0.6 * baseline + 0.4 * self.item_means[target_idx])
        return self._clip(baseline + numerator / denominator)

    def recommend(self, user_id, top_n=10):
        if user_id not in self.user_to_idx:
            return self.popular_items(top_n)
        seen = {item_id for item_id, _ in self.user_rated_items[user_id]}
        candidates = [
            item_id for item_id in self.items
            if item_id not in seen
            and self.item_popularity.get(item_id, 0) >= self.min_candidate_popularity
        ]
        scored = [(item_id, self.predict(user_id, item_id)) for item_id in candidates]
        scored.sort(key=lambda x: (x[1], self.item_popularity.get(x[0], 0)), reverse=True)
        return scored[:top_n]

    def popular_items(self, top_n):
        return [(item_id, self.item_mean(item_id))
            for item_id, _ in sorted(self.item_popularity.items(),
                key=lambda x: (x[1], self.item_mean(x[0])), reverse=True)[:top_n]]

    def user_mean(self, user_id):
        if user_id not in self.user_to_idx:
            return self.global_mean
        return float(self.user_means[self.user_to_idx[user_id]])

    def item_mean(self, item_id):
        if item_id not in self.item_to_idx:
            return self.global_mean
        return float(self.item_means[self.item_to_idx[item_id]])

    def _clip(self, value):
            return min(5.0, max(1.0, float(value)))


In [21]:
class MFRecommender:
    def __init__(self, n_factors=20, lr=0.005, reg=0.02, n_epochs=20,
                 min_candidate_popularity=5, seed=42):
        self.n_factors = n_factors
        self.lr = lr
        self.reg = reg
        self.n_epochs = n_epochs
        self.min_candidate_popularity = min_candidate_popularity
        self.seed = seed

    def fit(self, rows):
        self.users = sorted({u for u, _, _, _ in rows})
        self.items = sorted({i for _, i, _, _ in rows})
        self.user_to_idx = {u: idx for idx, u in enumerate(self.users)}
        self.item_to_idx = {i: idx for idx, i in enumerate(self.items)}
        n_users, n_items = len(self.users), len(self.items)
        self.user_rated_items = defaultdict(list)
        self.item_popularity = defaultdict(int)
        rating_sum = rating_count = 0
        for user_id, item_id, rating, _ in rows:
            self.user_rated_items[user_id].append((item_id, rating))
            self.item_popularity[item_id] += 1
            rating_sum += rating
            rating_count += 1
        self.global_mean = rating_sum / rating_count if rating_count > 0 else 3.0
        rng = np.random.default_rng(self.seed)
        self.U = rng.normal(0, 0.01, (n_users, self.n_factors)).astype(np.float32)
        self.V = rng.normal(0, 0.01, (n_items, self.n_factors)).astype(np.float32)
        self.b_u = np.zeros(n_users, dtype=np.float32)
        self.b_i = np.zeros(n_items, dtype=np.float32)
        train_list = [(self.user_to_idx[u], self.item_to_idx[i], r) for u, i, r, _ in rows]
        rng_shuffle = random.Random(self.seed)
        for epoch in range(self.n_epochs):
            rng_shuffle.shuffle(train_list)
            sq_err = 0.0
            for uidx, iidx, rating in train_list:
                pred = (self.global_mean + self.b_u[uidx] + self.b_i[iidx]
                        + float(self.U[uidx] @ self.V[iidx]))
                err = rating - pred
                sq_err += err ** 2
                u_old = self.U[uidx].copy()
                self.U[uidx] += self.lr * (err * self.V[iidx] - self.reg * self.U[uidx])
                self.V[iidx] += self.lr * (err * u_old - self.reg * self.V[iidx])
                self.b_u[uidx] += self.lr * (err - self.reg * self.b_u[uidx])
                self.b_i[iidx] += self.lr * (err - self.reg * self.b_i[iidx])
            if (epoch + 1) % 5 == 0:
                print(f"  [MF] epoch {epoch+1:>2}/{self.n_epochs}  train RMSE={math.sqrt(sq_err/len(train_list)):.4f}")
        return self

    def predict(self, user_id, item_id):
        has_u = user_id in self.user_to_idx
        has_i = item_id in self.item_to_idx
        if not has_u and not has_i:
            return self._clip(self.global_mean)
        if not has_u:
            return self._clip(self.global_mean + self.b_i[self.item_to_idx[item_id]])
        if not has_i:
            return self._clip(self.global_mean + self.b_u[self.user_to_idx[user_id]])
        uidx = self.user_to_idx[user_id]
        iidx = self.item_to_idx[item_id]
        return self._clip(self.global_mean + self.b_u[uidx] + self.b_i[iidx]
                          + float(self.U[uidx] @ self.V[iidx]))

    def recommend(self, user_id, top_n=10):
        if user_id not in self.user_to_idx:
            return self.popular_items(top_n)
        seen = {item_id for item_id, _ in self.user_rated_items[user_id]}
        candidates = [
            item_id for item_id in self.items
            if item_id not in seen
            and self.item_popularity.get(item_id, 0) >= self.min_candidate_popularity
        ]
        scored = [(item_id, self.predict(user_id, item_id)) for item_id in candidates]
        scored.sort(key=lambda x: (x[1], self.item_popularity.get(x[0], 0)), reverse=True)
        return scored[:top_n]

    def popular_items(self, top_n=10):
        sorted_items = sorted(self.item_popularity.keys(),
            key=lambda item_id: (self.item_popularity[item_id], self.item_mean(item_id)), reverse=True)
        return [(item_id, self.item_mean(item_id)) for item_id in sorted_items[:top_n]]

    def item_mean(self, item_id):
        if item_id not in self.item_to_idx:
            return self.global_mean
        return self._clip(self.global_mean + self.b_i[self.item_to_idx[item_id]])

  
    def _clip(self, value):
            return min(5.0, max(1.0, float(value)))


In [ ]:
class EnsembleRecommender:
    def __init__(self, knn_model, mf_model, weight_knn=0.5, min_candidate_popularity=5):
        self.knn = knn_model
        self.mf = mf_model
        self.w_knn = weight_knn
        self.w_mf = 1.0 - weight_knn
        self.min_candidate_popularity = min_candidate_popularity
        self.items = sorted(set(knn_model.items) | set(mf_model.items))
        self.item_popularity = mf_model.item_popularity
        self.user_rated_items = defaultdict(set)
        for uid, rated in knn_model.user_rated_items.items():
            for item_id, _ in rated:
                self.user_rated_items[uid].add(item_id)
        for uid, rated in mf_model.user_rated_items.items():
            for item_id, _ in rated:
                self.user_rated_items[uid].add(item_id)
        self.users = sorted(set(knn_model.users) | set(mf_model.users))
        self.user_to_idx = {u: idx for idx, u in enumerate(self.users)}

    def predict(self, user_id, item_id):
        return min(5.0, max(1.0, self.w_knn * self.knn.predict(user_id, item_id)
                                + self.w_mf  * self.mf.predict(user_id, item_id)))

    def recommend(self, user_id, top_n=10):
        seen = self.user_rated_items.get(user_id, set())
        candidates = [
            item_id for item_id in self.items
            if item_id not in seen
            and self.item_popularity.get(item_id, 0) >= self.min_candidate_popularity
        ]
        scored = [(item_id, self.predict(user_id, item_id)) for item_id in candidates]
        scored.sort(key=lambda x: (x[1], self.item_popularity.get(x[0], 0)), reverse=True)
        return scored[:top_n]


def rating_metrics(model, rows):
    sq_errors, abs_errors = [], []
    for user_id, item_id, rating, _ in rows:
        pred = model.predict(user_id, item_id)
        sq_errors.append((rating - pred) ** 2)
        abs_errors.append(abs(rating - pred))
    return {
        "rmse": math.sqrt(sum(sq_errors) / len(sq_errors)),
        "mae": sum(abs_errors) / len(abs_errors),
    }


def topn_metrics(model, eval_rows, top_n=10, relevant_threshold=4.0):
    known_users = getattr(model, "user_to_idx", {})
    eval_users = sorted({u for u, _, _, _ in eval_rows if u in known_users})
    relevant_by_user = defaultdict(set)
    for user_id, item_id, rating, _ in eval_rows:
        if rating >= relevant_threshold:
            relevant_by_user[user_id].add(item_id)
    all_recommended = set()
    novelty_scores = []
    precisions, recalls = [], []
    total_interactions = sum(model.item_popularity.values())
    for user_id in eval_users:
        recs = model.recommend(user_id, top_n=top_n)
        rec_items = [item_id for item_id, _ in recs]
        all_recommended.update(rec_items)
        for item_id in rec_items:
            pop = model.item_popularity.get(item_id, 1)
            novelty_scores.append(-math.log2(pop / total_interactions))
        relevant = relevant_by_user.get(user_id, set())
        if relevant:
            hits = len(set(rec_items) & relevant)
            precisions.append(hits / top_n)
            recalls.append(hits / len(relevant))
    precision = sum(precisions) / len(precisions) if precisions else 0.0
    recall = sum(recalls) / len(recalls) if recalls else 0.0
    f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0.0
    return {
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "coverage": len(all_recommended) / max(len(model.items), 1),
        "novelty": sum(novelty_scores) / len(novelty_scores) if novelty_scores else 0.0,
        "users_evaluated": len(eval_users),
    }


def tune_knn(train_rows, valid_rows, k_values, shrinkage, min_pop):
    results = []
    for k in k_values:
        print(f"  [KNN] K={k}")
        model = ItemKNNRecommender(k=k, shrinkage=shrinkage, min_candidate_popularity=min_pop).fit(train_rows)
        m = rating_metrics(model, valid_rows)
        results.append((k, m["rmse"], m["mae"]))
        print(f"         RMSE={m['rmse']:.4f}  MAE={m['mae']:.4f}")
    return min(results, key=lambda x: x[1]), results


def tune_mf(train_rows, valid_rows, factor_values, lr, reg, n_epochs, min_pop):
    results = []
    for n_factors in factor_values:
        print(f"  [MF] n_factors={n_factors}")
        model = MFRecommender(n_factors=n_factors, lr=lr, reg=reg, n_epochs=n_epochs,
                              min_candidate_popularity=min_pop).fit(train_rows)
        m = rating_metrics(model, valid_rows)
        results.append((n_factors, m["rmse"], m["mae"]))
        print(f"         RMSE={m['rmse']:.4f}  MAE={m['mae']:.4f}")
    return min(results, key=lambda x: x[1]), results


def tune_ensemble_weight(knn_model, mf_model, valid_rows, weight_steps=11, min_pop=5):
    results = []
    for i in range(weight_steps):
        w_knn = round(i / (weight_steps - 1), 1)
        ens = EnsembleRecommender(knn_model, mf_model, weight_knn=w_knn, min_candidate_popularity=min_pop)
        m = rating_metrics(ens, valid_rows)
        results.append((w_knn, m["rmse"], m["mae"]))
        print(f"  w_knn={w_knn:.1f} w_mf={1-w_knn:.1f}  RMSE={m['rmse']:.4f}  MAE={m['mae']:.4f}")
    return min(results, key=lambda x: x[1]), results


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--data-dir", default="ml-100k")
    parser.add_argument("--output-dir", default="saved_models")
    parser.add_argument("--k-values", default="20,40,80")
    parser.add_argument("--factor-values", default="10,20,50")
    parser.add_argument("--shrinkage", type=float, default=25.0)
    parser.add_argument("--lr", type=float, default=0.005)
    parser.add_argument("--reg", type=float, default=0.02)
    parser.add_argument("--n-epochs", type=int, default=20)
    parser.add_argument("--min-candidate-popularity", type=int, default=5)
    args = parser.parse_args(args=[])

    data_dir = Path(args.data_dir)
    output_dir = Path(args.output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    base_rows = read_ratings(data_dir / "ua.base")
    titles = read_items(data_dir / "u.item")
    inner_train_rows, valid_rows = split_validation_from_base(base_rows)
    k_values = [int(v.strip()) for v in args.k_values.split(",") if v.strip()]
    factor_values = [int(v.strip()) for v in args.factor_values.split(",") if v.strip()]

    print("=" * 60)
    print(f"ua.base: {len(base_rows):,} / inner train: {len(inner_train_rows):,} / validation: {len(valid_rows):,}")
    print("=" * 60)

    print("\n[KNN] K 튜닝")
    (best_k, _, _), knn_results = tune_knn(inner_train_rows, valid_rows, k_values, args.shrinkage, args.min_candidate_popularity)
    print(f"  → 최적 K = {best_k}")

    print("\n[MF] n_factors 튜닝")
    (best_factors, _, _), mf_results = tune_mf(inner_train_rows, valid_rows, factor_values, args.lr, args.reg, args.n_epochs, args.min_candidate_popularity)
    print(f"  → 최적 n_factors = {best_factors}")

    print("\n[KNN] 전체 재학습")
    final_knn = ItemKNNRecommender(k=best_k, shrinkage=args.shrinkage,
                                   min_candidate_popularity=args.min_candidate_popularity).fit(base_rows)
    print("[MF] 전체 재학습")
    final_mf = MFRecommender(n_factors=best_factors, lr=args.lr, reg=args.reg,
                             n_epochs=args.n_epochs, min_candidate_popularity=args.min_candidate_popularity).fit(base_rows)

    print("\n[Ensemble] 가중치 탐색")
    tmp_knn = ItemKNNRecommender(k=best_k, shrinkage=args.shrinkage,
                                 min_candidate_popularity=args.min_candidate_popularity).fit(inner_train_rows)
    tmp_mf = MFRecommender(n_factors=best_factors, lr=args.lr, reg=args.reg,
                           n_epochs=args.n_epochs, min_candidate_popularity=args.min_candidate_popularity).fit(inner_train_rows)
    (best_w_knn, _, _), weight_results = tune_ensemble_weight(tmp_knn, tmp_mf, valid_rows, min_pop=args.min_candidate_popularity)
    print(f"  → 최적 w_knn={best_w_knn:.1f}  w_mf={1-best_w_knn:.1f}")

    final_ensemble = EnsembleRecommender(final_knn, final_mf, weight_knn=best_w_knn,
                                         min_candidate_popularity=args.min_candidate_popularity)

    model_bundle = {
        "knn": final_knn,
        "mf": final_mf,
        "ensemble": final_ensemble,
        "titles": titles,
        "best_k": best_k,
        "best_factors": best_factors,
        "best_w_knn": best_w_knn,
        "knn_validation_results": knn_results,
        "mf_validation_results": mf_results,
        "ensemble_weight_results": weight_results,
    }
    save_path = output_dir / "models.pkl"
    with open(save_path, "wb") as f:
        pickle.dump(model_bundle, f)
    print(f"\n저장 완료: {save_path}")


if __name__ == "__main__":
    main()

main()

ua.base: 90,570 / inner train: 81,917 / validation: 8,653

[KNN] K 튜닝
  [KNN] K=20
         RMSE=0.9772  MAE=0.7498
  [KNN] K=40
         RMSE=0.9588  MAE=0.7397
  [KNN] K=80
         RMSE=0.9458  MAE=0.7335
  → 최적 K = 80

[MF] n_factors 튜닝
  [MF] n_factors=10
  [MF] epoch  5/20  train RMSE=0.9404
  [MF] epoch 10/20  train RMSE=0.9252
  [MF] epoch 15/20  train RMSE=0.9199
  [MF] epoch 20/20  train RMSE=0.9169
         RMSE=0.9467  MAE=0.7447
  [MF] n_factors=20
  [MF] epoch  5/20  train RMSE=0.9404
